In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# 1. Carregar os dados (Ajuste o caminho absoluto se necessário)
caminho_arquivo = r'C:\Users\othav\Desktop\repositorio BovDB\Create Datasets\Dataset 2026 WIN\dados_indicadores.csv'
df = pd.read_csv(caminho_arquivo, parse_dates=['datetime'])
df = df.dropna(subset=['trend']).reset_index(drop=True)

# 2. Engenharia de Features (Tornando o preço relativo)
df['Bollinger_%b'] = (df['close'] - df['Bollinger_Lower']) / (df['Bollinger_Upper'] - df['Bollinger_Lower'] + 1e-8)

colunas_ma = ['SMA_3', 'EMA_3', 'SMA_5', 'EMA_5', 'SMA_7', 'EMA_7', 'SMA_9', 'EMA_9', 'SMA_11', 'EMA_11']
for col in colunas_ma:
    if col in df.columns:
        df[col] = (df['close'] - df[col]) / df[col]

# 3. Divisão Temporal Estrita (Treino e Teste)
train_start, train_end = '2026-01-01', '2026-03-31'
test_start, test_end = '2026-04-01', '2026-06-30'

treino = df[(df['datetime'] >= train_start) & (df['datetime'] <= train_end)].copy()
teste = df[(df['datetime'] >= test_start) & (df['datetime'] <= test_end)].copy()

# 4. Limpeza de colunas absolutas
colunas_para_remover = [
    'datetime', 'date', 'close', 'open', 'low', 'high', 
    'volume', 'average', 'amount_stock', 'id_ticker', 'business',
    'Bollinger_Mid', 'Bollinger_Upper', 'Bollinger_Lower'
]
treino.drop(columns=colunas_para_remover, inplace=True, errors='ignore')
teste.drop(columns=colunas_para_remover, inplace=True, errors='ignore')

# Separação de X e y
X_train = treino.drop(columns=['trend'])  
y_train = treino['trend']  

X_test = teste.drop(columns=['trend'])  
y_test = teste['trend'] 

# 5. Normalização Resistente a Outliers
scaler = QuantileTransformer(output_distribution='uniform', random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Validação Cruzada Temporal (TimeSeriesSplit) com Balanceamento Interno
tscv = TimeSeriesSplit(n_splits=5) 
val_scores = []

print("Iniciando TimeSeriesSplit Cross-Validation...")
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_scaled)):
    X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_tr, y_val = y_train.iloc[train_idx].values, y_train.iloc[val_idx].values

    # BALANCEAMENTO (Oversampling): Duplicar a classe minoritária apenas no bloco de treino
    X_tr_df = pd.DataFrame(X_tr)
    X_tr_df['trend'] = y_tr
    
    max_class_size = X_tr_df['trend'].value_counts().max()
    classe_0 = X_tr_df[X_tr_df['trend'] == 0].sample(max_class_size, replace=True, random_state=42)
    classe_1 = X_tr_df[X_tr_df['trend'] == 1].sample(max_class_size, replace=True, random_state=42)
    
    treino_bal = pd.concat([classe_0, classe_1]).sample(frac=1, random_state=42)
    X_tr_bal = treino_bal.drop(columns=['trend']).values
    y_tr_bal = treino_bal['trend'].values

    # Parametrização Otimizada da MLP (Sem early_stopping para forçar o aprendizado)
    mlp_cv = MLPClassifier(
        hidden_layer_sizes=(100, 50), 
        activation='relu',           
        solver='adam',
        alpha=0.001,                 
        learning_rate_init=0.0005,
        early_stopping=False,         # Desativado para a rede não desistir rápido
        max_iter=300,                 # Obriga a treinar bastante
        random_state=42
    )
    mlp_cv.fit(X_tr_bal, y_tr_bal)
    
    y_val_pred = mlp_cv.predict(X_val)
    acc = accuracy_score(y_val, y_val_pred)
    val_scores.append(acc)
    print(f"Fold {fold+1} - Acurácia na Validação: {acc:.4f}")

# 7. Treinamento do Modelo Final
print("\nTreinando Modelo Final...")
# Balancear todo o conjunto de treino
X_train_df = pd.DataFrame(X_train_scaled)
X_train_df['trend'] = y_train.values

max_class_size_final = X_train_df['trend'].value_counts().max()
classe_0_final = X_train_df[X_train_df['trend'] == 0].sample(max_class_size_final, replace=True, random_state=42)
classe_1_final = X_train_df[X_train_df['trend'] == 1].sample(max_class_size_final, replace=True, random_state=42)

treino_final_bal = pd.concat([classe_0_final, classe_1_final]).sample(frac=1, random_state=42)
X_train_bal_final = treino_final_bal.drop(columns=['trend']).values
y_train_bal_final = treino_final_bal['trend'].values

final_model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.0005,
    early_stopping=False,
    max_iter=300,
    random_state=42
)
final_model.fit(X_train_bal_final, y_train_bal_final)

# 8. Previsões Finais
train_final_preds = final_model.predict(X_train_scaled)
test_preds = final_model.predict(X_test_scaled)

# 9. Resultados
print(f"\nAcurácia Média na Validação (TimeSeries CV): {np.mean(val_scores):.4f}")
print(f"Acurácia Final no Conjunto de TESTE OOS: {accuracy_score(y_test, test_preds):.4f}")

# 10. Visualização das Matrizes
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(confusion_matrix(y_train, train_final_preds), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title("Matriz - Treino (Jan a Mar)")

sns.heatmap(confusion_matrix(y_test, test_preds), annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title("Matriz - Teste Final OOS (Abr a Jun)")

plt.tight_layout()
plt.show()

print("\nRelatório de Classificação (Teste):")
print(classification_report(y_test, test_preds))

Iniciando TimeSeriesSplit Cross-Validation...


C:\Users\othav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 1 - Acurácia na Validação: 0.5000


C:\Users\othav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 2 - Acurácia na Validação: 0.5033


C:\Users\othav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 3 - Acurácia na Validação: 0.5028


C:\Users\othav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Fold 4 - Acurácia na Validação: 0.5009
